# 03 · Plant Disease Detection: MobileNetV2 (Google Colab)

Upload this notebook to Colab (**File → Upload notebook**).

**Dataset:** PlantVillage, 38 classes (14 crops, healthy and diseased leaves), about 54,000 colour images.
**Model:** MobileNetV2 (ImageNet weights) with a new classification head. Stage 1 trains the head with the base frozen; stage 2 fine-tunes the top 30 layers. Early stopping in both stages.
**Outputs** (saved to Google Drive → `MyDrive/agri-dss/`):
`disease_mnv2.keras`, `class_names.json`, `disease_metrics.json`, `disease_confusion_matrix.png`, `disease_training_curves.png`.
Copy the first two into `ml-service/models/` and the rest into `reports/`.

### Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. A Kaggle account. Kaggle → Settings → API → **Create New Token** downloads `kaggle.json`.

## 1. Check the GPU and TensorFlow version

In [ ]:
import tensorflow as tf, sys
print("TensorFlow:", tf.__version__, "| Python:", sys.version.split()[0])
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE. Enable it via Runtime → Change runtime type → T4 GPU")
!nvidia-smi --query-gpu=name,memory.total --format=csv || true

## 2. Mount Google Drive (where the trained model is saved)

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/agri-dss")
OUT.mkdir(parents=True, exist_ok=True)
print("Outputs will be saved to", OUT)

## 3. Kaggle credentials
**Option A (recommended):** add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's **🔑 Secrets** panel (left sidebar) and enable notebook access.
**Option B:** run the cell and upload your `kaggle.json` when prompted.

In [ ]:
import os, json
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    print("Using Kaggle credentials from Colab Secrets")
except Exception:
    from google.colab import files
    print("Upload kaggle.json")
    uploaded = files.upload()
    creds = json.loads(uploaded["kaggle.json"])
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = creds["username"], creds["key"]

## 4. Download PlantVillage from Kaggle (~2 GB, takes a few minutes)

In [ ]:
!pip -q install kaggle
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p /content/plantvillage --unzip -q
!ls /content/plantvillage

In [ ]:
# Find the colour-image folder that holds the 38 class sub-folders (e.g. "Tomato___Late_blight")
def find_class_root(base: Path) -> Path:
    candidates = [p for p in base.rglob("*") if p.is_dir()
                  and sum(1 for c in p.iterdir() if c.is_dir() and "___" in c.name) >= 38]
    colour = [p for p in candidates if "color" in str(p).lower()]
    return (colour or candidates)[0]

DATA_DIR = find_class_root(Path("/content/plantvillage"))
n_images = sum(1 for _ in DATA_DIR.rglob("*.[jJpP][pPnN]*[gG]"))
print("Class root:", DATA_DIR, "| images:", n_images)

## 5. Build tf.data pipelines
Stratification is approximate: a seeded 80/20 split of the full set, with the 20% part halved into **validation (10%)** and **test (10%)**.
Images are 224×224, RGB, raw 0-255 pixels. Rescaling happens **inside** the model, so the API can feed raw pixels.

In [ ]:
import json, numpy as np
IMG_SIZE, BATCH, SEED = (224, 224), 32, 42
AUTOTUNE = tf.data.AUTOTUNE

train_ds, holdout_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=0.2, subset="both", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH, label_mode="int", interpolation="bilinear")
class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
assert NUM_CLASSES == 38, f"expected 38 classes, found {NUM_CLASSES}"

holdout_batches = holdout_ds.cardinality().numpy()
val_ds = holdout_ds.take(holdout_batches // 2)
test_ds = holdout_ds.skip(holdout_batches // 2)

train_ds = train_ds.shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds, test_ds = val_ds.prefetch(AUTOTUNE), test_ds.prefetch(AUTOTUNE)
print("batches -> train:", train_ds.cardinality().numpy(), "val:", val_ds.cardinality().numpy(),
      "test:", test_ds.cardinality().numpy())
print(class_names[:5], "...")

## 6. Model: augmentation + MobileNetV2 (frozen) + new head

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")  # active only during training

base = keras.applications.MobileNetV2(input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,), name="image")           # raw 0-255 RGB
x = augment(inputs)
x = layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_preprocess")(x)  # -> [-1, 1], as MobileNetV2 expects
x = base(x, training=False)                                          # keep BatchNorm in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
model = keras.Model(inputs, outputs, name="disease_mnv2")
model.summary(show_trainable=True)

## 7. Stage 1: train the classification head (base frozen)

In [ ]:
ckpt = str(OUT / "disease_mnv2_checkpoint.keras")
def callbacks(patience):
    return [
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy", save_best_only=True),
    ]

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks(3))

## 8. Stage 2: fine-tune the top 30 layers of MobileNetV2
A low learning rate keeps the pretrained features from being destroyed. BatchNorm layers stay frozen.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
print("trainable layers in base:", sum(l.trainable for l in base.layers))

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
start = len(hist1.history["loss"])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=start + 10, initial_epoch=start, callbacks=callbacks(4))

## 9. Training curves

In [ ]:
import matplotlib.pyplot as plt
h = {k: hist1.history[k] + hist2.history[k] for k in ("accuracy", "val_accuracy", "loss", "val_loss")}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ("accuracy", "loss")):
    ax.plot(h[metric], color="#2a78d6", linewidth=2, label="train")
    ax.plot(h["val_" + metric], color="#eb6834", linewidth=2, label="validation")
    ax.axvline(start - 0.5, color="#52514e", linestyle="--", linewidth=1)
    ax.text(start - 0.4, ax.get_ylim()[1], " fine-tuning →", va="top", fontsize=9, color="#52514e")
    ax.set_title(metric.capitalize(), loc="left", fontweight="bold"); ax.set_xlabel("Epoch")
    ax.grid(color="#e5e4e0"); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(OUT / "disease_training_curves.png", dpi=150); plt.show()

## 10. Evaluate on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support, confusion_matrix
y_true, y_prob = [], []
for xb, yb in test_ds:
    y_prob.append(model.predict(xb, verbose=0)); y_true.append(yb.numpy())
y_true, y_prob = np.concatenate(y_true), np.concatenate(y_prob)
y_pred = y_prob.argmax(1)

acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
top3 = np.mean([t in row for t, row in zip(y_true, np.argsort(y_prob, 1)[:, -3:])])
low_conf = float(np.mean(y_prob.max(1) < 0.5))
print(f"Test accuracy {acc:.4f} | macro P {p:.4f} R {r:.4f} F1 {f1:.4f} | top-3 {top3:.4f} | "
      f"share below 0.5 confidence {low_conf:.4f} (n={len(y_true)})")
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3, zero_division=0))

In [ ]:
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
cm = confusion_matrix(y_true, y_pred)
short = [c.replace("___", ": ").replace("_", " ")[:32] for c in class_names]
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm, cmap=LinearSegmentedColormap.from_list("b", ["#f4f8fd", "#a9c8f0", "#2a78d6", "#0d3c7a"]),
            xticklabels=short, yticklabels=short, square=True, cbar_kws={"shrink": 0.6, "label": "Images"}, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title(f"MobileNetV2: confusion matrix on test set (n={len(y_true)}, accuracy {acc:.3f})", loc="left", fontweight="bold")
fig.tight_layout(); fig.savefig(OUT / "disease_confusion_matrix.png", dpi=130); plt.show()

## 11. Export the model + class names to Drive

In [ ]:
from datetime import datetime, timezone
model.save(OUT / "disease_mnv2.keras")
(OUT / "class_names.json").write_text(json.dumps(class_names, indent=2))
metrics = {
    "model_version": f"disease-mnv2-v1-{datetime.now(timezone.utc):%Y%m%d}",
    "tensorflow_version": tf.__version__,
    "dataset": {"name": "PlantVillage (color)", "images": int(n_images), "classes": NUM_CLASSES,
                "split": "80/10/10 train/val/test, seed 42"},
    "training": {"stage1_epochs": len(hist1.history["loss"]), "stage2_epochs": len(hist2.history["loss"]),
                 "fine_tuned_layers": 30, "image_size": list(IMG_SIZE), "batch_size": BATCH},
    "test": {"accuracy": round(float(acc), 4), "precision_macro": round(float(p), 4),
             "recall_macro": round(float(r), 4), "f1_macro": round(float(f1), 4),
             "top3_accuracy": round(float(top3), 4), "share_below_0_5_confidence": round(low_conf, 4)},
    "per_class": {k: v for k, v in report.items() if k in class_names},
}
(OUT / "disease_metrics.json").write_text(json.dumps(metrics, indent=2))
!ls -lh "{OUT}" 

## 12. Sanity check: reload the saved model and predict a few test images

In [ ]:
reloaded = keras.models.load_model(OUT / "disease_mnv2.keras")
xb, yb = next(iter(test_ds))
pr = reloaded.predict(xb[:6], verbose=0)
fig, axes = plt.subplots(1, 6, figsize=(18, 3.6))
for ax, img, true, prob in zip(axes, xb[:6], yb[:6], pr):
    ax.imshow(img.numpy().astype("uint8")); ax.axis("off")
    ax.set_title(f"{class_names[prob.argmax()].split('___')[-1][:18]}\n{prob.max():.2f} (true: {class_names[true].split('___')[-1][:12]})", fontsize=8)
plt.show()
assert np.allclose(pr, model.predict(xb[:6], verbose=0), atol=1e-4), "reloaded model differs!"
print("Reloaded model matches.")

## 13. Next steps
1. Download `disease_mnv2.keras` and `class_names.json` from **Drive → MyDrive/agri-dss/** (or run the cell below).
2. Copy them into `agri-dss/ml-service/models/`.
3. Copy `disease_metrics.json`, `disease_confusion_matrix.png` and `disease_training_curves.png` into `agri-dss/reports/`.
4. Locally: `ml-service\.venv\Scripts\python -m pip install -r requirements-tf.txt`, then restart the ML service. `GET /health` should show `disease: loaded`.

The local TensorFlow version should be the **same or newer** than the Colab version printed in step 1.

In [ ]:
from google.colab import files
for name in ("disease_mnv2.keras", "class_names.json", "disease_metrics.json"):
    files.download(str(OUT / name))